# LARRY FlowMap tutorial data

This notebook no longer tries to recreate the old package-version-sensitive embedding. It loads the saved FlowMap tutorial object used by the current geometry API and exports the embedding coordinates for figure notebooks that need a flat table.

In [ ]:
from pathlib import Path
import os
import sys

NOTEBOOK_DIR = Path.cwd()
FIG_DIR = NOTEBOOK_DIR / "figures" / "larry"
FIG_DIR.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("NUMBA_CACHE_DIR", str((NOTEBOOK_DIR / ".numba_cache").resolve()))
os.environ.setdefault("MPLCONFIGDIR", str((NOTEBOOK_DIR / ".mpl_cache").resolve()))

LOCAL_FLOWMAP_SRC = Path("/Users/jh/Projects/flowmap/src")
if LOCAL_FLOWMAP_SRC.exists() and str(LOCAL_FLOWMAP_SRC) not in sys.path:
    sys.path.insert(0, str(LOCAL_FLOWMAP_SRC))

DATA_PATH = (NOTEBOOK_DIR / "../data/flowmap_manuscript/larry/tutorial_larry_flowmap_data.joblib").resolve()
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing {DATA_PATH}. Put the tutorial_larry_flowmap_data.joblib source data file there before running."
    )

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

from flowmap.geometry import FixedPointAnalyzer, GeneGradientAnalyzer, LagrangianPathOptimizer
from flowmap.plot import plot_velocity_stream
from flowmap.utils import load_dataset

data = load_dataset(DATA_PATH)
emb = data.embedder
labels = data.obs["state_info"].astype(str).to_numpy()

X_gene = data.layers["spliced"]
V_gene = data.layers["velocity"]
emb.n_spline_points = None
emb.gene_names = np.asarray(data.var_names)

print(data)
print("Embedding:", emb.X_emb.shape, "Velocity:", emb.V_emb.shape)

In [ ]:
INTERMEDIATE_DIR = NOTEBOOK_DIR / "intermediate_data"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

embedding = pd.DataFrame({
    "dim1": emb.X_emb[:, 0],
    "dim2": emb.X_emb[:, 1],
    "velocity_dim1": emb.V_emb[:, 0],
    "velocity_dim2": emb.V_emb[:, 1],
    "state_info": labels,
})
out_csv = INTERMEDIATE_DIR / "larry_flowmap_embedding_from_tutorial.csv"
out_npy = INTERMEDIATE_DIR / "larry_flowmap_embedding_from_tutorial.npy"
embedding.to_csv(out_csv, index=False)
np.save(out_npy, emb.X_emb)
print(out_csv)
print(out_npy)
embedding.head()